# SWYNEX Task 1 — Dataset Exploration & Preprocessing

**Internship:** Machine Learning Internship  
**Dataset:** Titanic passenger dataset  
**Objective:** Explore the data, inspect and handle missing values, encode categorical variables, and prepare features for model training.

**Source:** https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv

Run all cells from top to bottom. An internet connection is required to download the dataset.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load the dataset

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(DATA_URL)
print("Dataset loaded successfully")
print("Shape:", df.shape)
display(df.head())

## 3. Initial exploration

In [ ]:
print("Columns:", df.columns.tolist())
print("\\nData types and non-null counts:")
df.info()
print("\\nNumerical summary:")
display(df.describe())
print("\\nCategorical summary:")
display(df.describe(include="object"))

## 4. Missing values and duplicates

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
display(missing_summary)
print("Duplicate rows:", df.duplicated().sum())

## 5. Exploratory data analysis

In [ ]:
df["survived"].value_counts().sort_index().plot(kind="bar", rot=0)
plt.title("Survival class distribution")
plt.xlabel("Survived (0 = No, 1 = Yes)")
plt.ylabel("Passenger count")
plt.tight_layout()
plt.show()

df["age"].hist(bins=25)
plt.title("Passenger age distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

display(df.groupby("sex", observed=False)["survived"].mean().to_frame("survival_rate"))

## 6. Select features and target

The `alive` column is excluded because it directly describes the target and would cause target leakage. The `deck` column is excluded because it has substantial missingness. The selected target is `survived`.

In [ ]:
selected_columns = [
    "survived", "pclass", "age", "sibsp", "parch",
    "fare", "sex", "embarked", "alone"
]
data = df[selected_columns].dropna(subset=["survived"]).copy()
X = data.drop(columns="survived")
y = data["survived"].astype(int)

numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked", "alone"]
print("Feature matrix:", X.shape)
print("Target vector:", y.shape)
print("Target classes:", sorted(y.unique()))

## 7. Split the data before fitting preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 8. Handle missing values and encode categorical variables

Preprocessing is fitted only on the training data to reduce data leakage. Numeric values use median imputation and standard scaling. Categorical values use most-frequent imputation and one-hot encoding.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

## 9. Verify processed features

In [ ]:
feature_names = preprocessor.get_feature_names_out()
train_array = X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed
test_array = X_test_processed.toarray() if hasattr(X_test_processed, "toarray") else X_test_processed

X_train_ready = pd.DataFrame(train_array, columns=feature_names, index=X_train.index)
X_test_ready = pd.DataFrame(test_array, columns=feature_names, index=X_test.index)
display(X_train_ready.head())
print("Missing values after preprocessing:", int(X_train_ready.isna().sum().sum()))

## 10. Export model-ready data

In [ ]:
output_dir = "processed_data"
os.makedirs(output_dir, exist_ok=True)
X_train_ready.to_csv(f"{output_dir}/X_train_processed.csv", index=False)
X_test_ready.to_csv(f"{output_dir}/X_test_processed.csv", index=False)
y_train.to_csv(f"{output_dir}/y_train.csv", index=False, header=["survived"])
y_test.to_csv(f"{output_dir}/y_test.csv", index=False, header=["survived"])
print("Processed feature and target files exported to:", output_dir)

## 11. Conclusion

The Titanic dataset was loaded and explored using shape, data types, descriptive statistics, missing-value analysis, and visualizations. A feature set and target were selected. The data was split into training and test sets before preprocessing. Numeric missing values were filled with the median and scaled. Categorical missing values were filled with the most frequent value and one-hot encoded.

`X_train_ready` and `X_test_ready` are prepared for a later machine-learning model. **No predictive model is trained in Task 1.**

**Reproducibility:** test size 20%, random state 42, preprocessing implemented with scikit-learn pipelines.